In [ ]:
!pip install bitsandbytes
!pip install google-api-python-client yt_dlp isodate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 102.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [ ]:
import os, cv2, tempfile
from moviepy.editor import *
import pandas as pd
import uuid
import requests
import cv2
import json
import torch
from transformers import LlavaNextVideoProcessor, LlavaNextVideoForConditionalGeneration

  if event.key is 'enter':



In [ ]:
from transformers import LlavaNextVideoProcessor, LlavaNextVideoForConditionalGeneration
from PIL import Image
import os
import json
import yt_dlp
import shutil
import isodate
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import files
import isodate
from googleapiclient.discovery import build
from datetime import datetime, timedelta
import isodate

In [ ]:


# Clave de API de YouTube (reemplázala con la tuya)
YOUTUBE_API_KEY = ""
COLAB_VIDEO_FOLDER = "/content/videos"

def customize_filters():
    """Allows users to modify default filtering keywords interactively."""
    default_positives = ["campaing", "trend", "aesthetic", "challenge", "reaction", "story time", "tutorial", "GRWM", "pov", "day in the life", "morning routine"]
    default_positives_dict = {1: "campaing", 2:"trend", 3:"aesthetic", 4:"challenge", 5:"reaction", 6:"story time", 7:"tutorial",
                              8:"GRWM", 9:"pov", 10:"day in the life", 11:"morning routine"}

    print("\n🛠️ Configuración de filtros de búsqueda 🛠️")
    print("🔹 Palabras clave POSITIVAS (aseguran que los videos sean relevantes):")
    print("   " + ", ".join(default_positives))

    print("\n🔸 Palabras clave NEGATIVAS (excluyen videos no deseados):")
    #print("   " + ", ".join([word.lstrip("-") for word in default_negatives]))

    category = int(input("\n¿Que categoria deseas? \n1.-campaing \n2.-trend \n3.-aesthetic \n4.-challenge \n5.-reaction \n6.-story time \n7.-tutorial \n8.-GRWM \n9.-pov \n10.-day in the life \n11.-morning routine:"))

    return default_positives_dict[category]# + default_negatives)


def get_start_of_week():
    today = datetime.utcnow()
    start_of_week = today - timedelta(days=today.weekday())  # Monday as start
    return start_of_week.replace(hour=0, minute=0, second=0, microsecond=0).isoformat("T") + "Z"

def search_youtube_shorts(query, max_results, min_duration, max_duration, custom_filters):
    """Searches for YouTube Shorts (videos ≤ 60s)."""
    youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

    refined_query = f"{query} {custom_filters}".strip()
    print(f"\n🔍 Searching for YouTube Shorts with filter: {refined_query}\n")

    published_after = get_start_of_week()

    request = youtube.search().list(
        q=refined_query,
        part="snippet",
        regionCode="US",
        maxResults=max_results * 5,  # Grab more for filtering
        type="video",
        publishedAfter=published_after,
    )

    response = request.execute()
    video_ids = [item["id"]["videoId"] for item in response["items"]]

    if not video_ids:
        return []

    # Get video details including duration
    request = youtube.videos().list(
        part="contentDetails,snippet",
        id=",".join(video_ids)
    )
    response = request.execute()

    shorts = []
    for item in response["items"]:
        duration_iso = item["contentDetails"]["duration"]
        duration_seconds = int(isodate.parse_duration(duration_iso).total_seconds())

        if duration_seconds <= 30:  # Short format videos
            title = item["snippet"]["title"]
            description = item["snippet"]["description"]

            # Optional: further verify it's likely a "Short" by keywords
            if "shorts" in title.lower() or "shorts" in description.lower():
                video_data = {
                    "title": title,
                    "video_id": item["id"],
                    "url": f"https://www.youtube.com/watch?v={item['id']}",
                    "description": description,
                    "published_at": item["snippet"]["publishedAt"],
                    "duration_seconds": duration_seconds
                }
                shorts.append(video_data)

            if len(shorts) >= max_results:
                break

    return shorts



def download_video(video_url, output_folder="videos"):
    """Descarga un video de YouTube en formato .mp4 usando yt-dlp."""
    os.makedirs(output_folder, exist_ok=True)
    ydl_opts = {
        "format": "best[ext=mp4]",
        "outtmpl": os.path.join(output_folder, "%(title)s.%(ext)s"),
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])

def zip_and_download():
    """Compresses all videos and metadata into a ZIP file and provides a download link."""
    zip_path = "/content/videos_download_brenda.zip"

    # Ensure metadata file exists before zipping
    metadata_path = os.path.join(COLAB_VIDEO_FOLDER, "videos_metadata.json")
    if not os.path.exists(metadata_path):
        print("⚠ Warning: Metadata file not found!")

    # Remove existing ZIP if it exists
    if os.path.exists(zip_path):
        os.remove(zip_path)

    # Create a ZIP file including videos and metadata
    shutil.make_archive(zip_path.replace(".zip", ""), 'zip', COLAB_VIDEO_FOLDER)

    # Download the ZIP file
    files.download(zip_path)


def extract_trend_video():
    """Main interactive function to fetch and download videos."""
    #animal = input("Ingrese la especie animal que desea buscar en YouTube: ").strip()
    animal = "tiktok"
    num_videos = int(input("Ingrese el número de videos que desea obtener: "))
    min_duration = int(input("Ingrese la duración mínima en segundos: "))
    max_duration = int(input("Ingrese la duración máxima en segundos: "))

    # Allow user to customize filters dynamically
    custom_filters = customize_filters()

    print(f"\nBuscando {num_videos} videos sobre '{animal}' en YouTube dentro del rango {min_duration}-{max_duration} segundos...\n")

    videos = search_youtube_shorts(animal, max_results=num_videos, min_duration=min_duration, max_duration=max_duration, custom_filters=custom_filters)

    # Check if the number of videos found is less than requested
    if len(videos) < num_videos:
        print(f"\n⚠ Solo se encontraron {len(videos)} videos en lugar de {num_videos}.")
        adjust = input("¿Desea intentar con filtros menos estrictos? (s/n): ").strip().lower()
        if adjust == "s":
            return main()  # Restart the function with new filters

    if not videos:
        print("❌ No se encontraron videos con la duración especificada.")
        return

    # Mostrar resultados
    top_1 = None
    for idx, video in enumerate(videos):
        if idx == 0:
          top1= video['title']
        print(f"{idx + 1}. {video['title']} ({video['duration_seconds']} segundos)")
        print(f"   URL: {video['url']}")
        print(f"   Publicado en: {video['published_at']}\n")

    # Ensure the videos folder exists before saving metadata
    os.makedirs(COLAB_VIDEO_FOLDER, exist_ok=True)
    metadata_path = os.path.join(COLAB_VIDEO_FOLDER, "videos_metadata.json")
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(videos, f, indent=4, ensure_ascii=False)

    # Descargar los videos
    for video in videos:
        print(f"⬇️ Descargando: {video['title']}...")
        download_video(video["url"])
        print("✅ Descarga completada.\n")

    print("📁 Proceso finalizado. Los videos y metadatos están guardados.")

    return os.path.join(COLAB_VIDEO_FOLDER, top1+".mp4")





In [ ]:
import shutil
import os


def clean_scraped_videos():
  #DELETES ALL VIDEOS AND METADATA FROM COLAB

  # Define the folder path
  video_folder = "/content/videos"

  # Remove all contents inside the folder
  shutil.rmtree(video_folder, ignore_errors=True)

  # Recreate the empty folder
  os.makedirs(video_folder, exist_ok=True)

  print("✅ Folder '/content/videos' has been cleared.")

  zipfile_path = "/content/videos_download.zip"
  if os.path.exists(zipfile_path):
      os.remove(zipfile_path)
      print("Zip file deleted successfully.")
  else:
      print("Zip file not found.")
  print(zipfile_path)

In [ ]:


def sample_video(video_path: str, video_out_path: str, sample_every = 30):
    print("sample every", sample_every)
    name = os.path.basename(video_path)
    base64Frames = []

    video = cv2.VideoCapture(video_path)
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

    idx_frame = 0
    samples = 0
    frame_list = []

    # Define the custom file name
    custom_filename = os.path.join(video_out_path, "{}_out.mp4".format(os.path.basename(video_path).split(".")[0]))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for mp4
    new_video_pose = cv2.VideoWriter(custom_filename, fourcc, 30.0, (width, height))

    while video.isOpened():
        success, frame = video.read()
        if not success:
            break

        if idx_frame % sample_every == 0:
            # Write the frame to the video
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frame_list.append(pil_img)

            new_video_pose.write(frame)
            samples +=1

        idx_frame += 1

    new_video_pose.release()
    print("Returning video with {} frames sampled".format(samples))
    return frame_list, custom_filename

In [ ]:

def load_video_analysis_model():
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model_id = "llava-hf/LLaVA-NeXT-Video-7B-hf"

  model = LlavaNextVideoForConditionalGeneration.from_pretrained(
      model_id,
      torch_dtype=torch.float16,
      low_cpu_mem_usage=True,
      load_in_4bit=True
  ).to(device)

  processor = LlavaNextVideoProcessor.from_pretrained(model_id)
  return model, processor

In [ ]:

def process_video(model, processor, video_frames):
  conversation = [
      {
          "role": "user",
          "content": [
              {"type": "text",
              "text": """Analyze the following TikTok video and extract the key attributes that describe its format, style, and potential as viral or branded content.
              Be very descriptive and critical
              Return the results as a JSON list with keys and values:
                      {
                      "visual_style": "Describe the visual style: cinematic, selfie, vlog, animation, POV, sketch, etc.",
                      "topic_summary": "What is the main topic or message of the video?",
                      "text_narration": "Summarize the key text or narration lines, preferably segmented by scene or screen, if they are the same, just show one",
                      "visual_assets_needed": "What kind of images or clips could be used to represent this video? (e.g., cityscape, person typing, books, nature, etc.)",
                      "audio_tone": "What is the tone of the audio? (calm, upbeat, dramatic, humorous, etc.)",
                      "audio_type": "Type of audio: voice-over, trending music, original sound, sound effect, silence",
                      "emotion_tone": "Main emotional tone: humor, nostalgia, surprise, tenderness, motivation, etc.",
                      "emotion_triggered": "What emotion does this video evoke?",
                      "trend_or_meme_reference": "Does it reference any trend or meme? Describe which one specifically, try your better guess",
                      "target_audience": "Describe what kind of audience this video seems intended for",
                      "audience_intent": "What would the viewer gain from this video? e.g., knowledge, entertainment, emotion, motivation"
                    } """},
              {"type": "video"},
              ],
      },
  ]
  prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

  inputs = processor(text=prompt, videos=video_frames, padding=True, return_tensors="pt").to(model.device)

  output = model.generate(**inputs, max_new_tokens=10000, do_sample=False)

  output_text = processor.decode(output[0][2:], skip_special_tokens=True)

  clean_text = output_text.split("json")[1].replace("```", "")

  json_output = json.loads(clean_text)
  return json_output


In [ ]:
def analyse_videos(model, processor, video, max_frames = 5,  sample_rate = 50, out_path = COLAB_VIDEO_FOLDER):
  video_frames, _ =  sample_video(video, out_path, sample_rate)
  print("Processing video...")
  metadata = process_video(model, processor, video_frames)
  print("Key features extracted!")
  return metadata



In [ ]:
model, processor = load_video_analysis_model()

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/70.2k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

In [ ]:
clean_scraped_videos()
video = extract_trend_video()
metadata = analyse_videos(model, processor, video, sample_rate = 30)
metadata

✅ Folder '/content/videos' has been cleared.
Zip file not found.
/content/videos_download.zip
Ingrese el número de videos que desea obtener: 1
Ingrese la duración mínima en segundos: 5
Ingrese la duración máxima en segundos: 30

🛠️ Configuración de filtros de búsqueda 🛠️
🔹 Palabras clave POSITIVAS (aseguran que los videos sean relevantes):
   campaing, trend, aesthetic, challenge, reaction, story time, tutorial, GRWM, pov, day in the life, morning routine

🔸 Palabras clave NEGATIVAS (excluyen videos no deseados):

¿Que categoria deseas? 
1.-campaing 
2.-trend 
3.-aesthetic 
4.-challenge 
5.-reaction 
6.-story time 
7.-tutorial 
8.-GRWM 
9.-pov 
10.-day in the life 
11.-morning routine:11

Buscando 1 videos sobre 'tiktok' en YouTube dentro del rango 5-30 segundos...


🔍 Searching for YouTube Shorts with filter: tiktok morning routine

1. get ready with me! #makeup #music #tiktok #grwm #grwmmakeup #getreadywithme #shorts #morningroutine (16 segundos)
   URL: https://www.youtube.com/watch

  warnings.warn(



Key features extracted!


{'visual_style': 'Cinematic',
 'topic_summary': 'Beauty and makeup tips',
 'text_narration': 'The video features a young woman demonstrating how to apply makeup, showcasing different products and techniques.',
 'visual_assets_needed': 'Makeup products, brushes, mirrors, and possibly a makeup artist',
 'audio_tone': 'Upbeat and motivational',
 'audio_type': 'Voice-over with trending music',
 'emotion_tone': 'Positive and motivational',
 'emotion_triggered': 'Empowerment and inspiration',
 'trend_or_meme_reference': 'Makeup tutorial',
 'target_audience': 'Young women interested in beauty and makeup',
 'audience_intent': 'Educational and entertaining'}

In [ ]:
brand_or_product = input("Talk about your brand or product: ")

Talk about your brand or product: a hat with the KFC Logo


# **API CALL TO : https://hailuoai.video/create**

In [ ]:
import os
import time
import requests
import json


api_key = ""




prompt = f"""Objective: Generate a compelling video  designed for viral potential, specifically targeting the current trend associated with the hashtag: {metadata["topic_summary"]}.

Core Strategy: Replicate key visual and conceptual elements identified in the top-performing video currently trending under # {metadata["topic_summary"]}.

Input Concepts (Derived from captioning/analysis of the top trending video):

visual style: {metadata["visual_style"]}
text narration: {metadata["text_narration"]}
visual assets_needed: {metadata["visual_assets_needed"]}
audio tone: {metadata["audio_tone"]}
audio type: {metadata["audio_type"]}
emotion tone: {metadata["emotion_tone"]}
emotion triggered: {metadata["emotion_triggered"]}
trend or meme reference: {metadata["trend_or_meme_reference"]}
target audience: {metadata["target_audience"]}
audience intent: {metadata["audience_intent"]}

Instructions for AI:

Incorporate Elements: Weave the listed Input Concepts naturally into the video's narrative, scenes, or visual focus.

Capture Trend Essence: Analyze the implied style, pacing, mood, camera angles (if applicable), and overall aesthetic of content typically found under #{metadata["topic_summary"]}.
Emulate these qualities in the generated video.

Viral Appeal: Optimize for engagement. The video should be visually interesting, potentially surprising or satisfying, and encourage sharing.
Focus on creating a strong hook within the first few seconds.

Brand Integration: Subtly and naturally incorporate the specified brand/product: {brand_or_product}.
Ensure the integration aligns with the trend's style and the video's core concepts, avoiding overly aggressive or out-of-place advertising.
The product/brand should feel like part of the scene or solution presented

Output: A video file incorporating the specified elements and targeting the identified trend."""

model = "T2V-01"
output_file_name = f"generated_video.mp4" #Please enter the save path for the generated video here

def invoke_video_generation()->str:
    print("-----------------Submit video generation task-----------------")
    url = "https://api.minimaxi.chat/v1/video_generation"
    payload = json.dumps({
      "prompt": prompt,
      "model": model
    })
    headers = {
      'authorization': 'Bearer ' + api_key,
      'content-type': 'application/json',
    }

    response = requests.request("POST", url, headers=headers, data=payload)
    print(response.text)
    task_id = response.json()['task_id']
    print("Video generation task submitted successfully, task ID.: "+task_id)
    return task_id

def query_video_generation(task_id: str):
    url = "https://api.minimaxi.chat/v1/query/video_generation?task_id="+task_id
    headers = {
      'authorization': 'Bearer ' + api_key
    }
    response = requests.request("GET", url, headers=headers)
    status = response.json()['status']
    if status == 'Preparing':
        print("...Preparing...")
        return "", 'Preparing'
    elif status == 'Queueing':
        print("...In the queue...")
        return "", 'Queueing'
    elif status == 'Processing':
        print("...Generating...")
        return "", 'Processing'
    elif status == 'Success':
        return response.json()['file_id'], "Finished"
    elif status == 'Fail':
        return "", "Fail"
    else:
        return "", "Unknown"


def fetch_video_result(file_id: str):
    print("---------------Video generated successfully, downloading now---------------")
    url = "https://api.minimaxi.chat/v1/files/retrieve?file_id="+file_id
    headers = {
        'authorization': 'Bearer '+api_key,
    }

    response = requests.request("GET", url, headers=headers)
    print(response.text)

    download_url = response.json()['file']['download_url']
    print("Video download link: " + download_url)
    with open(output_file_name, 'wb') as f:
        f.write(requests.get(download_url).content)
    print("THe video has been downloaded in："+os.getcwd()+'/'+output_file_name)


if __name__ == '__main__':
    task_id = invoke_video_generation()
    print("-----------------Video generation task submitted -----------------")
    while True:
        time.sleep(10)

        file_id, status = query_video_generation(task_id)
        if file_id != "":
            fetch_video_result(file_id)
            print("---------------Successful---------------")
            break
        elif status == "Fail" or status == "Unknown":
            print("---------------Failed---------------")
            break

-----------------Submit video generation task-----------------
{"task_id":"265060575813774","base_resp":{"status_code":0,"status_msg":"success"}}
Video generation task submitted successfully, task ID.: 265060575813774
-----------------Video generation task submitted -----------------
...Preparing...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
...Generating...
---------------Video generated successfully, downloading now---------------
{"file":{"file_id":265060855160906,"bytes":0,"created_at":1746265303,"filename":"output.mp4","purpose":"video_generation","download_url":"https://public-cdn-video-data-algeng.oss-cn-wulanchabu.aliyuncs.com/inference_output%2Fvideo%2F2025-05-03%2Fa7147f56-2f9b-4010-98ec-ff2a36278356%2Foutput.mp4?Expires=1746297731&OSSAccessKeyId=LTAI5tAmwsjSaaZVA6cEFAUu&Signature=2XR